---
title: "Die Gallerie (mit Code)"
jupyter: python3
#code-fold: true
execute:
  echo: true
  output: asis
lightbox: 
  match: auto
  effect: fade
  desc-position: bottom
  loop: true
highlight-style: atom-one
code-tools: true
code-fold: true
---

In [13]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pprint 
from pathlib import Path
from collections import defaultdict

print('<link rel="stylesheet" type="text/css" href="gallery.css">')
def query_WB(endpoint, query):

    # Initialize the wrapper
    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    # Fetch and parse results
    results = sparql.query().convert()

    return results 

def get_data(results):

    grouped = defaultdict(lambda: defaultdict(lambda: {"label": "", "descr": "", "photos": [], "wikibase_url": ""}))
    for result in results["results"]["bindings"]:
        item_url = result["item"]["value"]
        label = result.get("itemLabel", {}).get("value", "")
        photo = result.get("photo", {}).get("value", "")
        descr = result.get("itemDescription", {}).get("value", "")
        room_url = result["room"]["value"]
        room = result["roomLabel"]["value"]
        photographer = result.get("creator", {}).get("value", "")

        key = (room, room_url)
        art = grouped[key][item_url]

        art["label"] = label
        art["descr"] = descr
        art["wikibase_url"] = item_url

        if photo and not any(photo == p['photo'] for p in art["photos"]):
            art["photos"].append({"photo": photo, "photographer": photographer})

    return grouped

def generate_output(grouped):
    for (room, room_url), artworks in grouped.items():
        print(f"""
## {room}\\index{{{room}}}

""")
        
        for art_label in sorted(artworks):
            art = artworks[art_label]

            supported_photos = [entry for entry in art["photos"] if is_supported_image(entry["photo"])]
            unsupported_photos = [entry for entry in art["photos"] if not is_supported_image(entry["photo"])]

            if supported_photos:
                print(''' 
::: {.flex}
                ''')
                for entry in supported_photos:
                    print(f"""
::: {{.photo-div}}  
![Photo: {entry['photographer']}  
Bildquelle: [{entry["photo"]}]({entry["photo"]})]({entry['photo']}){{group="photos"}}
:::
""")
                print(''' 
:::
                      ''')

            for entry in unsupported_photos:
                print()


        print("\\newpage")


def is_supported_image(path):
    ext = Path(path).suffix.lower()
    return ext in [".jpg", ".jpeg", ".png"]





<link rel="stylesheet" type="text/css" href="gallery.css">


In [14]:
def make_book():

    endpoint_url = "https://query.kewl.org/sparql"

    query = """
PREFIX wd: <https://wikibase.kewl.org/entity/>
PREFIX wdt: <https://wikibase.kewl.org/prop/direct/>
PREFIX p: <https://wikibase.kewl.org/prop/>
PREFIX ps: <https://wikibase.kewl.org/prop/statement/>
PREFIX pq: <https://wikibase.kewl.org/prop/qualifier/>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX bd: <http://www.bigdata.com/rdf#>

SELECT DISTINCT ?castle ?item ?itemLabel ?castleLabel ?photo ?itemDescription ?room ?roomLabel ?creator WHERE {
  ?castle wdt:P4 ?anychild .   # Any item that is ever a parent (at any depth)

  ?item wdt:P3+ ?castle ;
        wdt:P3 ?room ;
        wdt:P1 wd:Q6 .
  ?item p:P6 ?statement .
  ?statement ps:P6 ?photo .
  OPTIONAL { ?statement pq:P11 ?creator. }
  FILTER NOT EXISTS { ?parent wdt:P4 ?castle . } 
  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "de"
  }
} ORDER BY ?roomLabel
    """


    results = query_WB(endpoint_url,query)
    grouped = get_data(results)
    generate_output(grouped)

make_book()


## Aktäon und Diana\index{Aktäon und Diana}


 
::: {.flex}
                

::: {.photo-div}  
![Photo: Gaasch, Uwe  
Bildquelle: [https://previous.bildindex.de/bilder/fmd10027229a.jpg](https://previous.bildindex.de/bilder/fmd10027229a.jpg)](https://previous.bildindex.de/bilder/fmd10027229a.jpg){group="photos"}
:::


::: {.photo-div}  
![Photo: Gaasch, Uwe  
Bildquelle: [https://previous.bildindex.de/bilder/fmd10027231a.jpg](https://previous.bildindex.de/bilder/fmd10027231a.jpg)](https://previous.bildindex.de/bilder/fmd10027231a.jpg){group="photos"}
:::

 
:::
                      
 
::: {.flex}
                

::: {.photo-div}  
![Photo: Gaasch, Uwe  
Bildquelle: [https://previous.bildindex.de/bilder/fmd10027229a.jpg](https://previous.bildindex.de/bilder/fmd10027229a.jpg)](https://previous.bildindex.de/bilder/fmd10027229a.jpg){group="photos"}
:::


::: {.photo-div}  
![Photo: Gaasch, Uwe  
Bildquelle: [https://previous.bildindex.de/bilder/fmd10027232a.jpg](https://previous.bildi